# Notebook 01 — Environment Setup & TargetDiff Installation

**Goals of this notebook:**
1. Mount Google Drive and clone the project repo
2. Install all Python dependencies
3. Clone TargetDiff and download its pretrained checkpoint
4. Verify RDKit, PyTorch, and the `src/` module all work

**End state:** your Drive contains a complete `egfr_diffusion/` folder with code, TargetDiff, and the model checkpoint.  All subsequent notebooks start by running the same setup cell and can re-use this state.

⚠️ **Before running: set `REPO_URL` in the first code cell to your GitHub URL.**

## Step 1 — Mount Drive, clone repo, install deps

This cell runs every time you open any notebook; it's idempotent (safe to re-run).

In [ ]:
# EDIT THIS with your own GitHub repo URL
REPO_URL = "https://github.com/Jonathan-Ye-1/egfr-drug-design-eecs6895-final-project.git"

# First-run bootstrap: clone the repo so we can import src.colab_init
import os, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
if not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Now use the helper to finish setup (pull latest, install deps, make dirs)
from src.colab_init import setup
PROJECT_ROOT = setup(repo_url=REPO_URL)

## Step 2 — Check GPU and package versions

In [ ]:
import torch, rdkit, numpy, pandas
print('PyTorch :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('RDKit   :', rdkit.__version__)
print('NumPy   :', numpy.__version__)
print('Pandas  :', pandas.__version__)

# Important: if CUDA is False, go to Runtime → Change runtime type → GPU

## Step 3 — Clone TargetDiff into Drive

In [ ]:
import os, subprocess
targetdiff_dir = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')
if not os.path.exists(targetdiff_dir):
    subprocess.run([
        'git', 'clone',
        'https://github.com/guanjq/targetdiff.git',
        targetdiff_dir
    ], check=True)
    print('TargetDiff cloned →', targetdiff_dir)
else:
    print('TargetDiff already present at', targetdiff_dir)

## Step 4 — Download TargetDiff pretrained checkpoint

One-time download (~300 MB). Will live in Drive so all future notebooks re-use it.

In [ ]:
import os
ckpt_dir = os.path.join(PROJECT_ROOT, 'external', 'targetdiff', 'checkpoints')
ckpt_path = os.path.join(ckpt_dir, 'pretrained_diffusion.pt')
os.makedirs(ckpt_dir, exist_ok=True)

if not os.path.exists(ckpt_path):
    !pip install -q gdown
    import gdown
    # Official TargetDiff checkpoint ID
    gdown.download(
        'https://drive.google.com/uc?id=1_BUWcHMQLbvOPbU4aYiDYcvF_0VEPjPZ',
        ckpt_path,
        quiet=False,
    )
else:
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print(f'Checkpoint already present ({size_mb:.0f} MB) → {ckpt_path}')

## Step 5 — Smoke test: RDKit + src module

In [ ]:
from rdkit import Chem
from rdkit.Chem import QED, Draw
from IPython.display import display

mol = Chem.MolFromSmiles('C(#N)c1cc2c(cc1OCC)NC(=O)c1cc(OCC)c(OCC)c(OC)c1N2')  # Erlotinib
print(f'Erlotinib QED = {QED.qed(mol):.3f}  (should be ~0.55)')
display(Draw.MolToImage(mol, size=(350, 220)))

from src import MolEvaluator, load_config
cfg = load_config('configs/targets.yaml')
print('\nTargets configured:', list(cfg['targets'].keys()))
print('Baselines configured:', list(cfg['baselines'].keys()))
print('\n✅ Setup complete. You can now run notebook 02.')

## What got saved to Drive

After this notebook finishes, your `/content/drive/MyDrive/egfr_diffusion/` contains:

```
egfr_diffusion/
├── src/  configs/  notebooks/    ← from GitHub
├── data/                         ← empty, ready for notebook 02
├── results/                      ← empty, ready for later notebooks
└── external/targetdiff/
    └── checkpoints/pretrained_diffusion.pt  ← ~300 MB
```

Close this notebook. Next time you open any notebook (02, 03, …), you only need to run its **first cell** (Step 1 above) — everything else is already in Drive.